__Delete this cell later__:

---

idea for structuring the text in notebook: We could do something like this:
 - Start by saying "we have done x, y & Z know ...", "based on the load data we can know do x..." and etc..
 - Do the next step/task.
 - Finish each thing with *Answer:* "We found that ..."

---

Things that needs check/changes:
- Idk if this requirements.txt is correct, if it is I thing its a good idea, this could also be done in a gitworkflow file, but maybe too much for now. 
- Check if source(s) and quotes are correct
- In the cleaning data section, idk about this price outlier removal - I just said everything under 500 remove. 

install the required packages for the project:
<br>
pip install -r requirements.txt

if the imports have changed:
<br>
pip freeze > requirements.txt

---

## Imports need for notebook 

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

## 1. Bussiness case / Problem statement

Across the world, cars are one of the most essential mode of transportaion. In som regions more then others, which leads to a high volume of vehicle transations. For example, according to S&P Global: <br><br>
&emsp;&emsp;*"[...] 2025 US auto sales in April expected to reach 1.49 million units [...]"*<br>
&emsp;&emsp;source: [automotive-insights](https://www.spglobal.com/automotive-insights/en/blogs/2025/02/us-auto-sales-2025) (last visited: 29-04-2025). 
<br>

Where there is a high number of cars being bought and sold, they will eventually end up at used car dealerships. According to [ibisworld](https://www.ibisworld.com/united-states/number-of-businesses/used-car-dealers/1004/) (last visited: 29-04-2025): There where <br>

&emsp;&emsp;*"[...] 130,152 Used Car Dealers in the US businesses as of 2023, an decrease of -0.6% from 2022."*
<br>

Based on this we can wonder how can these used car dealerships price their cars?
What factors inpacts the pricing of a used car? And can machine learning be used to give a accurate price prediction?

## 2. Data selection and preparation:

This section will include:
- Loading the data
- Cleaning the data

This exam project is based on the dataset: [Car Prices Dataset](https://www.kaggle.com/datasets/sidharth178/car-prices-dataset?select=train.csv) from kaggle. 

### 2.1 Load data

In [2]:
base_path = '../data/'

df = pd.read_csv(f'{base_path}dataset.csv', sep=',', header=0)

In [3]:
df.sample(5)

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
18042,45511745,16778,585,HYUNDAI,Elantra,2013,Sedan,No,Petrol,1.8,70000 km,4.0,Tiptronic,Front,04-May,Left wheel,Silver,0
2784,45760004,7840,-,OPEL,Astra,1998,Goods wagon,No,Petrol,1.6,230000 km,4.0,Manual,Front,04-May,Left wheel,Blue,4
9730,45654295,3,900,KIA,Sportage,2015,Jeep,No,Petrol,2.4,50345 km,4.0,Tiptronic,4x4,04-May,Left wheel,Blue,12
14594,45766527,31047,1053,MERCEDES-BENZ,E 350,2014,Sedan,Yes,Diesel,3.5,132349 km,6.0,Automatic,4x4,04-May,Left wheel,Grey,12
409,45615670,9722,473,CHEVROLET,Cruze,2014,Sedan,No,Petrol,1.4 Turbo,121000 km,4.0,Tiptronic,Front,04-May,Left wheel,Green,10


In [4]:
df.shape

(19237, 18)

In [5]:
pd.set_option('display.float_format', '{:.2f}'.format) # Set float format to 2 decimal places, to avoid exponential notation

df.describe()

,ID,Price,Prod. year,Cylinders,Airbags
count,19237.00,19237.00,19237.00,19237.00,19237.00
mean,45576535.89,18555.93,2010.91,4.58,6.58
std,936591.42,190581.27,5.67,1.20,4.32
min,20746880.00,1.00,1939.00,1.00,0.00
25%,45698374.00,5331.00,2009.00,4.00,4.00
50%,45772308.00,13172.00,2012.00,4.00,6.00
75%,45802036.00,22075.00,2015.00,4.00,12.00
max,45816654.00,26307500.00,2020.00,16.00,16.00


__Explanation:__

We have loaded the data using pandas `read_csv` function, where we have specified the separator to separate by sep = ',' and with header = 0 to instruct that the first row of the csv file is a header, wich will become the columns in the dataframe.

After the data has been loaded, we verify that the data has been loadded properly by sampling 5 ranomd rows of the data, by using the `.sample` function: 

We use the command `.shape` to comefirm that is the right size of that we expect, as there have been sated on kaggle (rows: 19237, cols: 18).


We can see that the data has been loaded properly.
<br>
Additionally this process also helps us to quickly get a view and idea of the data we are going to work with. 
<br>
We can first of all see that the `ID` column is not need beacuse the datafram has its own bulid in index. 
<br>
Secondly we can see that the `Doors` column looks a bit strange, beacause it containes 3 opstions: '04-May', '02-Mar' or '>5', which couold be a mistake in the data, so this has to be checked more in detail. 
<br>
Thirdly we can see that when `Levy` don't have a value its just a '-'. 
<br>
Lastly we can also see that the `Price` has an unrealistic min value of 1.00 and a bit outlier of 26307500.00 as max value, more then likely this should just be removed.

### 2.2 Cleaning- and Preprocessing Data

##### 2.2.1 NaN values

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

__Explanation:__

By using the `.info()` function, we can see that there are no null values in the data; however when we load the data in, we could see that for column: `Levy`, had '-' instead of NaN. Additionally when using the `.info()` function, we can also see that the dtypes appear to be all most only objects, these should be converted before to right type at the end for this cleaning process at least.

---

##### 2.2.2 Levy column

In [7]:
df_dash_count = df['Levy'].where(df['Levy'] == '-').count()

print(f"Number of '-' in Levy in train set: {df_dash_count}")

Number of '-' in Levy in train set: 5819


In [8]:
levy_avg = np.sum(df['Levy'].where(df['Levy'] != '-').astype(float) / df.shape[0])

print(f"Average Levy in train set: {levy_avg}")

df['Levy'] = df['Levy'].replace('-', int(levy_avg))

Average Levy in train set: 632.5286687113374


__Explanation:__

We have exchanged the '-' with the average ´Levy´ value, which is 632 approximately.

---

##### 2.2.3 Price column

In [9]:
low_outliers_custom = df[df['Price'] < 500]
high_outliers_custom = df[df['Price'] > 500000]

high_outliers_custom

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
1225,45795524,627220,632,MERCEDES-BENZ,G 65 AMG 63AMG,2020,Jeep,Yes,Petrol,6.3 Turbo,0 km,8.00,Tiptronic,4x4,04-May,Left wheel,Black,12
8541,45761204,872946,2067,LAMBORGHINI,Urus,2019,Universal,Yes,Petrol,4,2531 km,8.00,Tiptronic,4x4,04-May,Left wheel,Black,0
16983,45812886,26307500,632,OPEL,Combo,1999,Goods wagon,No,Diesel,1.7,99999 km,4.00,Manual,Front,02-Mar,Left wheel,Blue,0


In [10]:
df = df[df['Price'] > 500]
df = df.drop(index=16983)

__Explanation:__

As we saw, when we used the descpribe function, the `Price` had an unrealistic min (1) and max (2.6 m) value, most likely outliers, even though max could be real. We will have a closer look at this.

We have come to that conclusion that the 2.6m car is more a mistake then the real price, and that cars under 500 should be removed.

---

##### 2.2.4 Doors column

In [11]:
df['Doors'].value_counts()

Doors
04-May    16695
02-Mar      757
>5          119
Name: count, dtype: int64

In [12]:
df['Doors'] = df['Doors'].replace({
    '04-May': '4',
    '02-Mar': '2',
    '>5': '5'
})

__Explanation:__

As we mentioned before in 2.1 load data, we observed that `Doors` had a wrong format, by mistake. We have used .value_counts() to get a full view over the different types of values present in the dataframe, so we can transform these into more usable values.

---

##### 2.2.5 Engine volume column

In [13]:
df['Engine volume'].value_counts()

Engine volume
2            3706
2.5          2051
1.8          1569
1.6          1413
1.5          1203
             ... 
5.4 Turbo       1
0.3 Turbo       1
5.2             1
5.8             1
1.1 Turbo       1
Name: count, Length: 107, dtype: int64

In [14]:
turbo_conut = df['Engine volume'].where(df['Engine volume'].str.contains("Turbo")).count()
print(f"Number of Turbo engines: {turbo_conut}, which is {(turbo_conut / df.shape[0]) * 100:.2f}% of the dataset")

Number of Turbo engines: 1896, which is 10.79% of the dataset


In [15]:
df['turbo'] = df['Engine volume'].apply(lambda x: 1 if 'Turbo' in x else 0)
df['Engine volume'] = df['Engine volume'].str.replace('Turbo', '').astype(float)

__Explanation:__

Right know this `Engine volume` is an object in the dataframe, but could just be a float; however these values in `Engine volume` contains non-numeric character ('Turbo'), so before we can convert to float, we have to remove the turbo part. Instead of just removing we look at how many there are turbo engines to see if it is worth to keep this information. Here we saw that there are 1931 turbo engines, which is 10% ish of the cars engines, that are turbo, so we decided to keep this information, due to i maybe have inpact on the price.

Additionally we have made a new column that representes if a engine is turbo or not (1 = turbo, 0 = not turbo), and we have removed 'Turbo' at the end of each occasion and lastly converted the `Engine volume` to a float, by using `.astype(float)`.

---

##### 2.2.6 Leather interior column

In [16]:
df['Leather interior'].value_counts()

Leather interior
Yes    12542
No      5029
Name: count, dtype: int64

In [17]:
df['Leather interior'] = df['Leather interior'].replace({'No': 0, 'Yes': 1})
df['Leather interior'].value_counts()

Leather interior
1    12542
0     5029
Name: count, dtype: int64

__Explanation:__

We have converted the `Leather interior` column to a binary value (1 = yes, 0 = no), by using the `replace` function. We have done this because it is easier to work with binary values in a machine learning model, and it is also easier to interpret it, because it only yes or no, if it where more complex and we had to use one hot encoding, it would be more difficult to interpret the results later on in further analysis and explanations.

---

##### 2.2.7 Fuel type, Drive wheels, Gear box type

In [18]:
df = pd.get_dummies(df, columns=['Fuel type', 'Drive wheels', 'Gear box type'], drop_first=True)

df.sample(5)

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Engine volume,Mileage,...,Fuel type_Hybrid,Fuel type_Hydrogen,Fuel type_LPG,Fuel type_Petrol,Fuel type_Plug-in Hybrid,Drive wheels_Front,Drive wheels_Rear,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator
1087,45816011,11133,1079,HYUNDAI,Elantra,2018,Sedan,1,2.00,40122 km,...,False,False,False,True,False,True,False,False,False,False
19158,44710924,36065,1018,BMW,X5,2011,Jeep,1,3.00,78000 km,...,False,False,False,True,False,False,False,False,True,False
1226,45730842,44134,730,SSANGYONG,Actyon,2016,Jeep,1,1.60,35398 km,...,False,False,False,False,False,True,False,False,False,False
8424,45797004,9722,1030,NISSAN,Note,2014,Hatchback,0,1.20,125000 km,...,False,False,False,True,False,True,False,False,False,False
7525,45656637,706,585,TOYOTA,Prius,2013,Hatchback,1,1.80,105328 km,...,True,False,False,False,False,True,False,False,False,False


__Explanation:__

As stated above its not as easy to convert types to a numeric value if there are more the 2 options, so we will use one hot encoding for these columns. We will be doing this by using the `get_dummies` function.

We use `df.sample(5)` to verify that the hot encoding has been executed. Additionally we see that we how have 27 columns do to the one hot encoding.

---

##### 2.2.8 Transformation of data types to numeric

In [19]:
df['Mileage'] = df['Mileage'].str.replace(' km', '')

df.rename(columns={'Mileage': 'Mileage_km'}, inplace=True)

to_numeric = ['Prod. year', 'Mileage_km', 'Levy', 'Doors', 'Leather interior']

df[to_numeric] = df[to_numeric].apply(pd.to_numeric, errors='coerce')

__Explanation:__

Every value in the `Mileage` column ends with 'km', we would rather just have the numric value, so it can be easier to work with. Therfore have we first of all removed the 'km' for each value and changed the name of the column to `Mileage_km`.

The columns: `Prod. year`, `Mileage_km`, `Levy`, `Doors` have been changed to numeric values, by applying the function pd.to_numeric() to each column, so that these columns can be used in training the models and making predictions.

---

##### 2.2.9 Droped columns 

In [20]:
df.drop(columns=['ID'], inplace=True)

__Explanation:__

As stated previously, we remove `ID` because we don't need two indexes, due to dataframe has an build in index and an additional is redundant.

---

#### 2.2.9 Verifying data after cleaning

In [21]:
df.sample(5)

,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Engine volume,Mileage_km,Cylinders,...,Fuel type_Hybrid,Fuel type_Hydrogen,Fuel type_LPG,Fuel type_Petrol,Fuel type_Plug-in Hybrid,Drive wheels_Front,Drive wheels_Rear,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator
16403,9408,632,BMW,320 Gran Turismo,2001,Sedan,0,2.20,111111,6.00,...,False,False,False,True,False,False,True,True,False,False
5014,5331,632,MITSUBISHI,Pajero IO,1998,Jeep,0,1.80,250000,4.00,...,False,False,False,True,False,False,False,False,False,False
12458,36231,642,HYUNDAI,Santa FE,2012,Jeep,1,2.00,68359,4.00,...,False,False,False,False,False,True,False,False,False,False
15252,37633,632,TOYOTA,Corolla se,2018,Sedan,1,1.80,18000,4.00,...,False,False,False,True,False,True,False,False,True,False
8024,44545,770,HYUNDAI,Tucson,2016,Jeep,1,1.70,56331,4.00,...,False,False,False,False,False,True,False,False,False,False


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17571 entries, 0 to 19235
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Price                     17571 non-null  int64  
 1   Levy                      17571 non-null  int64  
 2   Manufacturer              17571 non-null  object 
 3   Model                     17571 non-null  object 
 4   Prod. year                17571 non-null  int64  
 5   Category                  17571 non-null  object 
 6   Leather interior          17571 non-null  int64  
 7   Engine volume             17571 non-null  float64
 8   Mileage_km                17571 non-null  int64  
 9   Cylinders                 17571 non-null  float64
 10  Doors                     17571 non-null  int64  
 11  Wheel                     17571 non-null  object 
 12  Color                     17571 non-null  object 
 13  Airbags                   17571 non-null  int64  
 14  turbo      

__Explanation:__

Just in case, we check if the values have been drop, have missing values and that we have the right data types, so we don't end with unexpected values in the data. 
<br>
We do this as we did after loading the data, by randomly sampling 5 rows from the datasets and using the info function to see null values and data types.

We can know comfirm that the values have infact been droped, have the right data types and that we have no missing values.

Lastly we save the cleaned data to a new csv file, so we can use it for the other part of the this project.

In [23]:
df.to_csv(f'{base_path}dataset_cleaned.csv', sep=',', index=False)

## 3 Data exploration and visualization:

This section will include:
- x
- y



--- 
Look at the data and try to understand it. What are the most important features?

Make some visualizations to understand the data better, by showing the distribution of the features and the target variable.

Try to find correlations between the features and the target variable, by making scatter plots and correlation matrices.

etc ...

## 4. Model Selection and training:

This section will include:
- x
- y

### 4.1 The selected model(s)

We have selected the following models for this project:
- Multiple Linear Regression
- Model 2

The reason for the selected models is as follows:
- Multiple Linear Regression: ..
- Model 2: This model is selected because ...

We are going to be using the following error metrics to evalaute the models:
- x
- y
- z

Additionaly we are going to be useing the grid search approach for ...

### 4.2 Data splitting into sets


In [24]:
# split the dataset into train, validation and test sets

### 4.3 Model training and validation

Based on the selected models we have trained the models using the training data. The training process is as follows:

"describe the training process here, that we are going to use and why that makes sence"

### 4.4 Model evaluation